# HW3 Applied: Zero-Shot Super Resolution (ZSSR)

**Course:** Modern Computer Vision (Technion, Spring 2026)

**Before you start**, read the paper:

> Shocher et al., ["Zero-Shot Super-Resolution using Deep Internal Learning"](https://openaccess.thecvf.com/content_cvpr_2018/papers/Shocher_Zero-Shot_Super-Resolution_Using_CVPR_2018_paper.pdf) (CVPR 2018)
>
> Project page: [http://www.wisdom.weizmann.ac.il/~vision/zssr/](http://www.wisdom.weizmann.ac.il/~vision/zssr/)

**Objective:** Implement ZSSR — a method that trains a small CNN on a *single image* at test time, exploiting the internal recurrence of patches within natural images to achieve super-resolution without any external training data.

**What you implement:** Two things only — the network architecture and the full train+evaluate pipeline. Everything else (utilities, visualization) is provided in `utils.py`.

**Submit** this notebook **with all outputs visible** (training curves, comparison images). Notebooks without outputs will lose points.

In [ ]:
# Your student ID (replace with your actual ID)
STUDENT_ID = "YOUR_ID_HERE"

# Setup and imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from utils import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

## Grading

Your grade is based entirely on **PSNR improvement over bicubic upsampling**.

**Base (50 pts):** Your `train_and_evaluate` runs correctly on the two provided test images and beats the bicubic baseline on both.

**PSNR score (up to 35 pts × 2 images = 70 pts):** We run your code on two additional unseen test images and measure the PSNR improvement *p* (in dB) over bicubic. The score per image is:

$$\text{score} = \min\!\Big(35,\;\Big\lfloor\frac{p^2}{2}\Big\rfloor\Big)$$

This rewards bigger improvements disproportionately — squeezing out an extra 0.1 dB at the top is harder than the first 0.1 dB above bicubic.

**Total: up to 120 pts.**

## The ZSSR Idea

Traditional super-resolution trains on large external datasets of LR-HR pairs. ZSSR takes a radically different approach: it trains a CNN on a **single test image** at test time.

### How it works

Given a low-resolution image **I_LR** that we want to upscale by factor *s*:

1. **Generate training pairs from the image itself:**
   - Crop a patch from **I_LR** — call it the "father"
   - Downscale that patch by factor *s* — call it the "son"
   - The pair (son → father) is one training example

2. **Train a CNN** to map son → father (i.e., to undo the downscaling)

3. **Apply the trained CNN** to the full **I_LR** to produce **I_SR**

### Why it works

Natural images have **internal patch recurrence** — small patches repeat across scales within the same image. By learning to upscale the image's own downscaled patches, the CNN learns priors specific to *this particular image*.

### Architecture

A simple and effective architecture:
- **Bicubic upsampling** of the input to the target resolution
- A stack of **convolutional layers** (e.g., 8 layers, 64 channels, 3×3 kernels)
- **Residual connection**: the network learns the *residual* (correction) on top of the bicubic upsampling

The residual design is crucial — it lets the network focus on learning the high-frequency details that bicubic interpolation misses.

## Part 1: The Model — ZSSRNet

Implement a CNN for zero-shot super-resolution. Your network should:
1. Accept a low-resolution input and a scale factor
2. Upsample to the target resolution
3. Produce a refined super-resolved output

The `forward` method receives `scale_factor` and optionally `target_size=(H, W)` so it knows the desired output resolution. Read the paper for architecture ideas — the original uses 8 conv layers with 64 channels.

In [ ]:
class ZSSRNet(nn.Module):
    """
    Zero-Shot Super Resolution Network.

    Takes a low-resolution image and a scale factor, produces a super-resolved output.

    Args:
        in_channels: number of image channels (3 for RGB)
        n_channels: number of intermediate feature channels (default: 64)
        n_layers: number of conv layers (default: 8)
        kernel_size: conv kernel size (default: 3)
    """

    def __init__(self, in_channels=3, n_channels=64, n_layers=8, kernel_size=3):
        super().__init__()
        raise NotImplementedError("Implement ZSSRNet")

    def forward(self, x, scale_factor=2, target_size=None):
        """
        Args:
            x: input tensor (B, C, H, W) or (C, H, W)
            scale_factor: upsampling factor (e.g., 4)
            target_size: optional (H, W) tuple for output spatial size.
                         If None, use input_size * scale_factor.
        Returns:
            Super-resolved output, same batch/channel dims as input.
        """
        raise NotImplementedError("Implement forward pass")

## Part 2: Train and Evaluate — `train_and_evaluate`

This is the core of ZSSR. Given a low-resolution image, you should:

1. **Create a `ZSSRNet` model**
2. **Train it on the input image itself:**
   - Randomly crop a patch from the LR image (the "father")
   - Downscale that crop by `scale_factor` to create the "son"
   - Train the model to map son → father
   - Repeat for many steps (each step uses a fresh random crop — there is no fixed dataset)
3. **Apply the trained model** to the full LR image to produce the SR output

### Key implementation details:
- **Crop first, then downscale** — never pre-downscale the whole image and try to match coordinates
- **L1 loss** tends to work better than L2 for super-resolution
- **Adam optimizer** with learning rate ~0.001
- The original paper uses **~3000 gradient steps** per image. You can experiment with fewer for speed, but more steps generally help.
- **Clamp output** to [0, 1] before returning

### What to return:
A single tensor — the super-resolved image of shape `(C, H_out, W_out)` where `H_out ≈ H_in * scale_factor`.

In [ ]:
def train_and_evaluate(image_lr, scale_factor=4, device='cpu'):
    """
    Train ZSSR on a single low-resolution image and return the super-resolved result.

    This function should:
    1. Create a ZSSRNet model
    2. Train it using pairs generated from image_lr itself
       (crop patches, downscale them, train model to undo the downscaling)
    3. Apply the trained model to the full image_lr
    4. Return the super-resolved output

    Args:
        image_lr: Low-resolution input image, tensor of shape (C, H, W), values in [0, 1]
        scale_factor: Integer upsampling factor (e.g., 4)
        device: 'cpu' or 'cuda'

    Returns:
        image_sr: Super-resolved image, tensor of shape (C, H*sf, W*sf),
                  values clamped to [0, 1]
    """
    raise NotImplementedError("Implement train_and_evaluate")

## Ideas for Improvement

Your grade depends on the PSNR improvement your method achieves over bicubic upsampling. Below are techniques from the paper and beyond that can boost performance. **Read the paper carefully** — most of these are described there in detail.

- **Residual learning** (Section 3 in paper): Instead of learning the full output, learn the *residual* on top of a bicubic upsampling. This makes optimization much easier — the network only needs to predict the missing high-frequency details.

- **Data augmentation** (Section 3): Apply geometric augmentations (4 rotations × 2 flips = 8-fold symmetry) to each training patch. This multiplies your effective training data.

- **Test-time geometric self-ensemble** (Section 3): At inference, apply all 8 augmentations to the LR image, super-resolve each, reverse the augmentation, and average/median the results.

- **Back-projection** (Section 3): After getting an initial SR estimate, enforce LR-consistency: downscale SR, compute the error vs. the original LR, upscale the error, and correct. This can be repeated iteratively.

- **Gradual SR — training data** (Section 3): Instead of training only on the (son, father) pair at scale *s*, add training pairs at intermediate scales (e.g., *s*^(1/2), *s*^(1/3), ...). This gives the network more diverse examples.

- **Gradual SR — sequential application** (Section 3): Instead of upscaling directly by ×4, upscale progressively (e.g., ×2 then ×2, or even finer steps like ×1.5 → ×1.5 → ×1.78). Each intermediate result can be added to the training set for the next stage.

- **Loss function**: L1 loss is standard, but you could try L2, a combination, or a gradient-domain loss.

- **Architecture tweaks**: More/fewer layers, wider channels, skip connections, different activations.

- **Learning rate scheduling**: Reduce LR during training (e.g., StepLR, CosineAnnealing).

None of these require separate cells — implement them inside your `ZSSRNet` and `train_and_evaluate` as you see fit.

## Run Your ZSSR

Load test images, run your pipeline, and visualize results. The `visualize_sr` function from `utils.py` creates a flickering GIF comparing your result with the bicubic baseline.

In [ ]:
# Load test images
test_images = load_test_images()
scale_factor = 4

for name, hr_image in test_images.items():
    print(f"\n{'='*60}")
    print(f"Processing {name}: {hr_image.shape}")
    print('=' * 60)

    # Create LR by downscaling HR
    lr_image = resize_bicubic(hr_image, scale_factor=1.0 / scale_factor)
    print(f"LR image: {lr_image.shape}")

    # Run ZSSR
    sr_image = train_and_evaluate(lr_image, scale_factor=scale_factor, device=device)
    print(f"SR image: {sr_image.shape}")

    # Visualize (compares SR vs bicubic, shows PSNR if HR is available)
    results = visualize_sr(sr_image, lr_image, hr_image=hr_image, title=name)
    if results:
        print(f"Bicubic PSNR: {results['PSNR_Bicubic']:.2f} dB")
        print(f"ZSSR PSNR:    {results['PSNR_ZSSR']:.2f} dB")
        print(f"Improvement:  {results['PSNR_ZSSR'] - results['PSNR_Bicubic']:+.2f} dB")